In [27]:
from langchain_classic.tools import retriever
from requests_oauthlib.compliance_fixes import ebay
! pip install -r ./requirements.txt

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

In [17]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite")

In [21]:
llm.invoke([HumanMessage(content = "Send me the digit '1'")])

AIMessage(content='1', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019da134-3481-7892-bd55-5b866d4880d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 1, 'total_tokens': 9, 'input_token_details': {'cache_read': 0}})

In [49]:
#Creating vector database and text splitter
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
persist_directory = "VectorStorage"
collection_name = "MCP_Server_Stuff"


if not os.path.exists(persist_directory):
   print(f"Creating directory {persist_directory}")
   os.makedirs(persist_directory)

text_splitter = RecursiveCharacterTextSplitter(
   chunk_size=100,
   chunk_overlap=20,
   length_function=len,
)


In [35]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2"
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6995.66it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
from langchain_community.document_loaders import TextLoader
#загрузка документов
loader = TextLoader("../client_side/sigma.txt", encoding="utf-8")

# Загружаем
documents = loader.load()
documents

[Document(metadata={'source': '../client_side/sigma.txt'}, page_content='Hello bebra')]

In [46]:
splitted_documents = text_splitter.split_documents(documents)

In [52]:

vectorstore = Chroma.from_documents(
   documents=splitted_documents,
   embedding=embeddings,
   persist_directory=persist_directory,
   collection_name=collection_name
)
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k":2})


In [54]:
retriever.invoke("Привет")

[Document(id='14a44461-7129-43bb-bf9a-397efb6b114f', metadata={'source': '../client_side/sigma.txt'}, page_content='Hello bebra'),
 Document(id='65639152-0b86-4188-a3dd-d16c078505c6', metadata={'source': '../client_side/sigma.txt'}, page_content='Hello bebra')]